# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we iterate over the record sets defined in the Croissant schema and list the associated fields and their `@id`. This will help us reference data entities precisely in future steps.

**Note:** All references are made using their `@id` as required by Croissant.

In [ ]:
# Explore available record sets and their fields
record_sets = dataset.metadata.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record set @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', 'N/A')}")
    if 'field' in rs:
        print("  Fields:")
        for f in rs['field']:
            field_id = f['@id'] if isinstance(f, dict) and '@id' in f else f
            print(f"    - {field_id}")
    print("---------------------------")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In this section, we will:
- Choose a main record set to extract (you may modify this to explore other sets later)
- Load its records and convert them into a pandas DataFrame
- Display its columns and preview the data

In [ ]:
# List all record set @id values
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
print('Available record set @id values:', record_set_ids)

# Example: Use the first record set (edit this variable if you want to explore others)
main_record_set_id = record_set_ids[0]
print(f"\nExtracting records from: {main_record_set_id}\n")

records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)
print('Columns:', df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

Below is an example using a numeric field (edit the field `@id` and threshold based on the field overview):

In [ ]:
# Pick a numeric field (replace with your schema's actual field @id, e.g. 'accession', 'coverage', etc.)
# We'll try to auto-detect a likely numeric field
possible_numeric_fields = [c for c in df.columns if df[c].dtype in ['int64', 'float64']]
print('Numeric candidate fields:', possible_numeric_fields)

if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]
else:
    # Fallback: try to coerce first column to numeric
    numeric_field = df.columns[0]
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

threshold = df[numeric_field].quantile(0.75) if df[numeric_field].notna().sum() else 0
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
print(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a categorical field (if present)
possible_group_fields = [c for c in df.columns if df[c].dtype == 'object' and c != numeric_field]
if possible_group_fields:
    group_field = possible_group_fields[0]
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"\nGrouped data by {group_field}:")
    print(grouped_df.head())
else:
    print('No categorical field found for grouping.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below we plot the distribution of the numeric field and, if available, boxplots for groups.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.show()

# If grouping field available, draw a boxplot
if possible_group_fields:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=45, ha='right')
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We successfully loaded and explored the Mass Spectrometry dataset using `mlcroissant` and pandas.
- Using record set and field `@id`s, we precisely referenced and extracted data.
- Simple exploratory analysis and visualizations revealed the structure and basic statistics of a key numeric field, grouped by possible categories.

Proceed with advanced statistics, machine learning experiments, or domain-specific analysis as desired.